# WiDS 2026 Wildfire Survival SuperStack


In [1]:
import os
import re
import json
import glob
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEEDS = [101, 202, 303]
HORIZONS = [12, 24, 48, 72]
FOLDS = 5
EPS = 1e-6
USE_EXTERNAL_PRIORS_IF_FOUND = True

KNOWN_PUBLIC_SCORES = {
    1: 0.96617,
    2: 0.96540,
    3: 0.96961,
    4: 0.97252,
    5: 0.97167,
    6: 0.97204,
    7: 0.97252,
    8: 0.97058,
    9: 0.95804,
    10: 0.97118,
    11: 0.96933,
    12: 0.96539,
    13: 0.96400,
    14: 0.95893,
    15: 0.96617,
    16: 0.96539,
    17: 0.97252,
}

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]


def find_first(filename):
    candidates = []
    for root in SEARCH_ROOTS:
        if os.path.exists(root):
            candidates.extend(glob.glob(os.path.join(root, "**", filename), recursive=True))
    candidates = sorted(set(candidates), key=lambda p: (len(p), p))
    return candidates[0] if candidates else None


def find_all_samplecsv_subs():
    paths = []
    for root in SEARCH_ROOTS:
        if os.path.exists(root):
            paths.extend(glob.glob(os.path.join(root, "**", "samplecsv_sub*.csv"), recursive=True))
    paths = sorted(set(paths), key=lambda p: (len(p), p))
    return paths


def clip01(x, lo=1e-6, hi=1 - 1e-6):
    return np.clip(np.asarray(x, dtype=float), lo, hi)


def sigmoid(x):
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-x))


def logit_clip(p):
    p = clip01(p)
    return np.log(p / (1.0 - p))


def signed_log1p(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))


def rank_norm(x):
    r = rankdata(np.asarray(x, dtype=float), method="average")
    return (r - 0.5) / len(r)


def harrell_c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    concordant = 0.0
    tied = 0.0
    comparable = 0.0
    n = len(time)
    for i in range(n):
        ti = time[i]
        ei = event[i]
        ri = risk[i]
        for j in range(i + 1, n):
            tj = time[j]
            ej = event[j]
            rj = risk[j]
            if ti < tj and ei == 1:
                comparable += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0
            elif tj < ti and ej == 1:
                comparable += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0
    if comparable == 0:
        return 0.5
    return (concordant + 0.5 * tied) / comparable


def horizon_labels(time, event, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    valid = (event == 1) | (time >= (horizon - 1e-12))
    y = ((event == 1) & (time <= horizon)).astype(int)
    return valid, y


def weighted_brier(time, event, probs_by_h):
    total = 0.0
    for h, w in [(24, 0.3), (48, 0.4), (72, 0.3)]:
        valid, y = horizon_labels(time, event, h)
        p = np.asarray(probs_by_h[h], dtype=float)
        total += w * np.mean((p[valid] - y[valid]) ** 2)
    return total


def hybrid_score(time, event, prob12, prob24, prob48, prob72):
    cidx = harrell_c_index(time, event, prob12)
    wb = weighted_brier(time, event, {24: prob24, 48: prob48, 72: prob72})
    score = 0.3 * cidx + 0.7 * (1.0 - wb)
    return score, cidx, wb


def make_feature_frame(df):
    out = df.copy()

    if "dist_min_ci_0_5h" in out.columns:
        dist = np.clip(out["dist_min_ci_0_5h"].astype(float), 0, None)
        out["dist_km"] = dist / 1000.0
        out["log1p_dist_min"] = np.log1p(dist)
        out["inv_dist_min"] = 1.0 / (1.0 + dist)

    if "dist_std_ci_0_5h" in out.columns:
        out["log1p_dist_std"] = np.log1p(np.clip(out["dist_std_ci_0_5h"].astype(float), 0, None))

    for col in [
        "dist_change_ci_0_5h",
        "dist_slope_ci_0_5h",
        "closing_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "projected_advance_m",
        "dist_accel_m_per_h2",
        "cross_track_component",
        "along_track_speed",
    ]:
        if col in out.columns:
            out[f"slog_{col}"] = signed_log1p(out[col])

    for col in [
        "area_first_ha",
        "area_growth_abs_0_5h",
        "radial_growth_m",
        "radial_growth_rate_m_per_h",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
    ]:
        if col in out.columns:
            out[f"log1p_{col}"] = np.log1p(np.clip(out[col].astype(float), 0, None))

    if {"dist_min_ci_0_5h", "projected_advance_m"}.issubset(out.columns):
        dist = np.clip(out["dist_min_ci_0_5h"].astype(float), 0, None)
        out["advance_frac"] = out["projected_advance_m"].astype(float) / (1.0 + dist)

    if {"dist_min_ci_0_5h", "closing_speed_m_per_h"}.issubset(out.columns):
        dist = np.clip(out["dist_min_ci_0_5h"].astype(float), 0, None)
        csp = out["closing_speed_m_per_h"].astype(float).to_numpy()
        out["closing_ratio"] = csp / (1.0 + dist)
        t_linear = np.where(csp > 1e-9, dist / csp, 1e6)
        out["log1p_time_to_contact_linear"] = np.log1p(np.clip(t_linear, 0, 1e6))
        out["positive_closing"] = (csp > 0).astype(int)

    if {"alignment_abs", "closing_speed_abs_m_per_h"}.issubset(out.columns):
        out["aligned_closing_abs"] = out["alignment_abs"].astype(float) * out["closing_speed_abs_m_per_h"].astype(float)

    if {"alignment_cos", "along_track_speed"}.issubset(out.columns):
        out["aligned_along_track"] = out["alignment_cos"].astype(float) * out["along_track_speed"].astype(float)

    if {"log1p_growth", "log1p_dist_min"}.issubset(out.columns):
        out["growth_over_distance"] = out["log1p_growth"].astype(float) / (1.0 + out["log1p_dist_min"].astype(float))

    if {"log1p_area_first", "log1p_dist_min"}.issubset(out.columns):
        out["size_over_distance"] = out["log1p_area_first"].astype(float) / (1.0 + out["log1p_dist_min"].astype(float))

    if {"dist_km", "alignment_abs", "closing_speed_m_per_h", "log1p_growth"}.issubset(out.columns):
        prox = 1.0 / (1.0 + out["dist_km"].astype(float))
        close_term = np.maximum(out["closing_speed_m_per_h"].astype(float), 0.0) / 250.0
        out["threat_score"] = prox * (1.0 + close_term) * (1.0 + out["alignment_abs"].astype(float)) * (1.0 + out["log1p_growth"].astype(float))
        out["threat_score_log"] = np.log1p(np.clip(out["threat_score"].astype(float), 0, None))

    if {"dist_min_ci_0_5h", "closing_speed_m_per_h"}.issubset(out.columns):
        out["close_fast_flag"] = ((out["dist_min_ci_0_5h"].astype(float) < 10000.0) & (out["closing_speed_m_per_h"].astype(float) > 0.0)).astype(int)

    if {"num_perimeters_0_5h", "low_temporal_resolution_0_5h"}.issubset(out.columns):
        out["single_or_lowres"] = ((out["num_perimeters_0_5h"].astype(float) <= 1.0) | (out["low_temporal_resolution_0_5h"].astype(float) > 0.0)).astype(int)

    if {"area_growth_abs_0_5h"}.issubset(out.columns):
        out["zero_growth_flag"] = (np.abs(out["area_growth_abs_0_5h"].astype(float)) < 1e-12).astype(int)

    if {"event_start_hour"}.issubset(out.columns):
        hr = out["event_start_hour"].astype(float)
        out["hour_sin"] = np.sin(2.0 * np.pi * hr / 24.0)
        out["hour_cos"] = np.cos(2.0 * np.pi * hr / 24.0)

    if {"event_start_dayofweek"}.issubset(out.columns):
        dw = out["event_start_dayofweek"].astype(float)
        out["dow_sin"] = np.sin(2.0 * np.pi * dw / 7.0)
        out["dow_cos"] = np.cos(2.0 * np.pi * dw / 7.0)

    if {"event_start_month"}.issubset(out.columns):
        mo = out["event_start_month"].astype(float)
        out["month_sin"] = np.sin(2.0 * np.pi * mo / 12.0)
        out["month_cos"] = np.cos(2.0 * np.pi * mo / 12.0)

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def impute_fit(X_train, X_valid, X_test):
    imp = SimpleImputer(strategy="median")
    Xtr = imp.fit_transform(X_train)
    Xva = imp.transform(X_valid)
    Xte = imp.transform(X_test)
    return Xtr, Xva, Xte


def constant_predictions(y_train, n_valid, n_test):
    c = float(np.mean(y_train)) if len(y_train) else 0.0
    return np.full(n_valid, c, dtype=float), np.full(n_test, c, dtype=float)


def fit_cb_binary(Xtr, ytr, Xva_all, Xte_all, seed):
    if len(np.unique(ytr)) < 2:
        return constant_predictions(ytr, len(Xva_all), len(Xte_all))
    pos = int(np.sum(ytr))
    neg = int(len(ytr) - pos)
    class_weights = [1.0, max(1.0, neg / max(pos, 1))]
    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=450,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=10.0,
        random_strength=1.0,
        bagging_temperature=0.3,
        class_weights=class_weights,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(Xtr, ytr)
    return model.predict_proba(Xva_all)[:, 1], model.predict_proba(Xte_all)[:, 1]


def fit_xgb_binary(Xtr, ytr, Xva_all, Xte_all, seed):
    if len(np.unique(ytr)) < 2:
        return constant_predictions(ytr, len(Xva_all), len(Xte_all))
    pos = int(np.sum(ytr))
    neg = int(len(ytr) - pos)
    scale_pos_weight = max(1.0, neg / max(pos, 1))
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=650,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=8.0,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        max_bin=256,
        random_state=seed,
        verbosity=0,
        n_jobs=-1,
    )
    model.fit(Xtr, ytr)
    return model.predict_proba(Xva_all)[:, 1], model.predict_proba(Xte_all)[:, 1]


def fit_lgb_binary(Xtr, ytr, Xva_all, Xte_all, seed):
    if len(np.unique(ytr)) < 2:
        return constant_predictions(ytr, len(Xva_all), len(Xte_all))
    pos = int(np.sum(ytr))
    neg = int(len(ytr) - pos)
    class_weight = {0: 1.0, 1: max(1.0, neg / max(pos, 1))}
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=650,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=-1,
        min_child_samples=10,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.3,
        reg_lambda=6.0,
        class_weight=class_weight,
        random_state=seed,
        verbosity=-1,
        n_jobs=-1,
    )
    model.fit(Xtr, ytr)
    return model.predict_proba(Xva_all)[:, 1], model.predict_proba(Xte_all)[:, 1]


def fit_cb_cox(Xtr, time_tr, event_tr, Xva_all, Xte_all, seed):
    y_signed = np.asarray(time_tr, dtype=float).copy()
    y_signed[np.asarray(event_tr, dtype=int) == 0] *= -1.0
    model = CatBoostRegressor(
        loss_function="Cox",
        iterations=500,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=10.0,
        random_strength=1.0,
        bagging_temperature=0.3,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(Xtr, y_signed)
    return model.predict(Xva_all), model.predict(Xte_all)


def fit_xgb_cox(Xtr, time_tr, event_tr, Xva_all, Xte_all, seed):
    y_signed = np.asarray(time_tr, dtype=float).copy()
    y_signed[np.asarray(event_tr, dtype=int) == 0] *= -1.0
    model = xgb.XGBRegressor(
        objective="survival:cox",
        eval_metric="cox-nloglik",
        n_estimators=700,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=8.0,
        tree_method="hist",
        max_bin=256,
        random_state=seed,
        verbosity=0,
        n_jobs=-1,
    )
    model.fit(Xtr, y_signed)
    return model.predict(Xva_all), model.predict(Xte_all)


def build_long_format(X, time, event, bins=(12, 24, 48, 72)):
    starts = [0] + list(bins[:-1])
    ends = list(bins)
    rows = []
    X = np.asarray(X)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    for i in range(len(X)):
        xi = X[i]
        t = time[i]
        e = event[i]
        for k, (s, en) in enumerate(zip(starts, ends)):
            if t <= s:
                break
            if e == 0 and t < en:
                break
            label = 1 if (e == 1 and s < t <= en) else 0
            feat = np.concatenate([
                xi,
                np.array([k, s, en, 0.5 * (s + en), np.log1p(en)], dtype=float),
            ])
            rows.append((feat, label))
            if label == 1:
                break
            if e == 0 and np.isclose(t, en):
                break
    if not rows:
        return None, None
    Z = np.vstack([r[0] for r in rows])
    y = np.asarray([r[1] for r in rows], dtype=int)
    return Z, y


def hazards_to_cuminc(haz):
    haz = clip01(haz, 1e-8, 1 - 1e-8)
    surv = np.cumprod(1.0 - haz, axis=1)
    return 1.0 - surv


def fit_discrete_hazard_cb(Xtr, time_tr, event_tr, Xva_all, Xte_all, seed):
    Ztr, ytr = build_long_format(Xtr, time_tr, event_tr)
    if Ztr is None or len(np.unique(ytr)) < 2:
        base_rate = float(np.mean((np.asarray(event_tr, dtype=int) == 1) & (np.asarray(time_tr, dtype=float) <= 24.0)))
        pred_valid = np.full((len(Xva_all), 4), base_rate, dtype=float)
        pred_test = np.full((len(Xte_all), 4), base_rate, dtype=float)
        return pred_valid, pred_test
    pos = int(np.sum(ytr))
    neg = int(len(ytr) - pos)
    class_weights = [1.0, max(1.0, neg / max(pos, 1))]
    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=400,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=10.0,
        random_strength=1.0,
        bagging_temperature=0.3,
        class_weights=class_weights,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(Ztr, ytr)

    def predict_ci(X):
        X = np.asarray(X, dtype=float)
        haz_cols = []
        starts = [0, 12, 24, 48]
        ends = [12, 24, 48, 72]
        for k, (s, en) in enumerate(zip(starts, ends)):
            block = np.concatenate(
                [
                    X,
                    np.full((len(X), 1), k, dtype=float),
                    np.full((len(X), 1), s, dtype=float),
                    np.full((len(X), 1), en, dtype=float),
                    np.full((len(X), 1), 0.5 * (s + en), dtype=float),
                    np.full((len(X), 1), np.log1p(en), dtype=float),
                ],
                axis=1,
            )
            haz_cols.append(model.predict_proba(block)[:, 1])
        haz = np.column_stack(haz_cols)
        return hazards_to_cuminc(haz)

    return predict_ci(Xva_all), predict_ci(Xte_all)


def fit_platt(raw_pred, y, valid_mask):
    raw_pred = np.asarray(raw_pred, dtype=float)
    y = np.asarray(y, dtype=int)
    valid_mask = np.asarray(valid_mask, dtype=bool)
    if valid_mask.sum() == 0:
        return ("const", 0.0)
    yy = y[valid_mask]
    if len(np.unique(yy)) < 2:
        return ("const", float(np.mean(yy)))
    model = LogisticRegression(C=0.5, solver="lbfgs", max_iter=2000, class_weight="balanced")
    model.fit(raw_pred[valid_mask].reshape(-1, 1), yy)
    return ("lr", model)


def apply_calibrator(calibrator, raw_pred):
    kind, obj = calibrator
    if kind == "const":
        return np.full(len(raw_pred), float(obj), dtype=float)
    return obj.predict_proba(np.asarray(raw_pred, dtype=float).reshape(-1, 1))[:, 1]


def fit_simplex_blend(Z, y):
    Z = np.asarray(Z, dtype=float)
    y = np.asarray(y, dtype=float)
    m = Z.shape[1]
    init = np.full(m, 1.0 / m, dtype=float)

    def objective(w):
        p = Z @ w
        reg = 1e-4 * np.mean((w - init) ** 2)
        return np.mean((p - y) ** 2) + reg

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(0.0, 1.0)] * m
    res = minimize(objective, init, method="SLSQP", bounds=bounds, constraints=constraints)
    if res.success:
        w = np.asarray(res.x, dtype=float)
    else:
        w = init.copy()
    w = np.clip(w, 0.0, None)
    if w.sum() <= 0:
        w = init.copy()
    else:
        w = w / w.sum()
    return w


def make_prob12_from_risk(risk, prob24):
    prob24 = clip01(prob24, 2e-6, 1.0 - 1e-6)
    rank01 = rank_norm(risk)
    scale = max(float(np.min(prob24)) - 1e-6, 1e-8)
    prob12 = rank01 * scale
    prob12 = np.minimum(prob12, prob24 - 1e-12)
    prob12 = np.clip(prob12, 0.0, 1.0)
    return prob12


def opinion_pool(internal, external, internal_weight=0.6):
    external_weight = 1.0 - internal_weight
    z = internal_weight * logit_clip(internal) + external_weight * logit_clip(external)
    return sigmoid(z)


def build_external_prior_blend(test_ids):
    paths = find_all_samplecsv_subs()
    if not paths:
        return None

    loaded = []
    id_df = pd.DataFrame({"event_id": test_ids})
    for path in paths:
        m = re.search(r"samplecsv_sub(\d+)\.csv$", os.path.basename(path))
        if not m:
            continue
        sub_no = int(m.group(1))
        score = KNOWN_PUBLIC_SCORES.get(sub_no)
        if score is None or score < 0.96900:
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        needed = {"event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"}
        if not needed.issubset(df.columns):
            continue
        try:
            df = id_df.merge(df[["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]], on="event_id", how="left", validate="1:1")
        except Exception:
            continue
        if df.isna().any().any():
            continue
        loaded.append((sub_no, score, path, df))

    if len(loaded) == 0:
        return None

    vecs = []
    for _, _, _, df in loaded:
        vecs.append(logit_clip(df[["prob_24h", "prob_48h", "prob_72h"]].to_numpy()).ravel())
    vecs = np.vstack(vecs)
    if len(loaded) > 1:
        corr = np.corrcoef(vecs)
        corr = np.nan_to_num(corr, nan=0.0)
    else:
        corr = np.ones((1, 1), dtype=float)

    raw_weights = []
    for i, (sub_no, score, _, _) in enumerate(loaded):
        if len(loaded) > 1:
            others = [corr[i, j] for j in range(len(loaded)) if j != i]
            mean_corr = float(np.mean(others)) if others else 0.0
        else:
            mean_corr = 0.0
        diversity = max(0.05, 1.0 - max(mean_corr, 0.0))
        perf = max(score - 0.96850, 0.0) ** 2
        raw_weights.append(perf * (0.50 + diversity))

    raw_weights = np.asarray(raw_weights, dtype=float)
    if raw_weights.sum() <= 0:
        raw_weights = np.full(len(loaded), 1.0 / len(loaded), dtype=float)
    else:
        raw_weights = raw_weights / raw_weights.sum()

    p12_rank = np.zeros(len(test_ids), dtype=float)
    z24 = np.zeros(len(test_ids), dtype=float)
    z48 = np.zeros(len(test_ids), dtype=float)
    z72 = np.zeros(len(test_ids), dtype=float)
    used = []

    for w, (sub_no, score, path_i, df) in zip(raw_weights, loaded):
        p12_rank += w * rank_norm(df["prob_12h"].to_numpy())
        z24 += w * logit_clip(df["prob_24h"].to_numpy())
        z48 += w * logit_clip(df["prob_48h"].to_numpy())
        z72 += w * logit_clip(df["prob_72h"].to_numpy())
        used.append({"submission": sub_no, "public_score": score, "weight": float(w), "path": path_i})

    out = {
        "risk_rank": rank_norm(p12_rank),
        "prob24": sigmoid(z24),
        "prob48": sigmoid(z48),
        "prob72": sigmoid(z72),
        "members": used,
    }
    return out


train_path = find_first("train.csv")
test_path = find_first("test.csv")
sample_path = find_first("sample_submission.csv")

if train_path is None or test_path is None or sample_path is None:
    raise FileNotFoundError("Could not locate train.csv, test.csv, and sample_submission.csv under /kaggle/input or working directories.")

print("train:", train_path)
print("test:", test_path)
print("sample:", sample_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

train_fe = make_feature_frame(train)
test_fe = make_feature_frame(test)

feature_cols = [c for c in train_fe.columns if c not in ["event_id", "time_to_hit_hours", "event"]]
X_train_df = train_fe[feature_cols].copy()
X_test_df = test_fe[feature_cols].copy()

time = train_fe["time_to_hit_hours"].to_numpy(dtype=float)
event = train_fe["event"].astype(int).to_numpy()

n_train = len(train_fe)
n_test = len(test_fe)

raw_prob_oof = {name: np.zeros((n_train, 4), dtype=float) for name in ["cb", "xgb", "lgb", "dh"]}
raw_prob_test = {name: np.zeros((n_test, 4), dtype=float) for name in ["cb", "xgb", "lgb", "dh"]}
raw_risk_oof = {name: np.zeros(n_train, dtype=float) for name in ["cox_cb", "cox_xgb"]}
raw_risk_test = {name: np.zeros(n_test, dtype=float) for name in ["cox_cb", "cox_xgb"]}

strata = event.copy()
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), strata), 1):
    print(f"Fold {fold}/{FOLDS}")

    Xtr_np, Xva_np, Xte_np = impute_fit(X_train_df.iloc[tr_idx], X_train_df.iloc[va_idx], X_test_df)
    time_tr = time[tr_idx]
    event_tr = event[tr_idx]

    for seed in SEEDS:
        risk_va, risk_te = fit_cb_cox(Xtr_np, time_tr, event_tr, Xva_np, Xte_np, seed)
        raw_risk_oof["cox_cb"][va_idx] += risk_va / len(SEEDS)
        raw_risk_test["cox_cb"] += risk_te / (len(SEEDS) * FOLDS)

        risk_va, risk_te = fit_xgb_cox(Xtr_np, time_tr, event_tr, Xva_np, Xte_np, seed)
        raw_risk_oof["cox_xgb"][va_idx] += risk_va / len(SEEDS)
        raw_risk_test["cox_xgb"] += risk_te / (len(SEEDS) * FOLDS)

        ci_va, ci_te = fit_discrete_hazard_cb(Xtr_np, time_tr, event_tr, Xva_np, Xte_np, seed)
        raw_prob_oof["dh"][va_idx] += ci_va / len(SEEDS)
        raw_prob_test["dh"] += ci_te / (len(SEEDS) * FOLDS)

    for h_idx, h in enumerate(HORIZONS):
        valid_tr, y_tr_all = horizon_labels(time_tr, event_tr, h)
        y_tr = y_tr_all[valid_tr]
        Xtr_h = Xtr_np[valid_tr]

        for seed in SEEDS:
            p_va, p_te = fit_cb_binary(Xtr_h, y_tr, Xva_np, Xte_np, seed)
            raw_prob_oof["cb"][va_idx, h_idx] += p_va / len(SEEDS)
            raw_prob_test["cb"][:, h_idx] += p_te / (len(SEEDS) * FOLDS)

            p_va, p_te = fit_xgb_binary(Xtr_h, y_tr, Xva_np, Xte_np, seed)
            raw_prob_oof["xgb"][va_idx, h_idx] += p_va / len(SEEDS)
            raw_prob_test["xgb"][:, h_idx] += p_te / (len(SEEDS) * FOLDS)

            p_va, p_te = fit_lgb_binary(Xtr_h, y_tr, Xva_np, Xte_np, seed)
            raw_prob_oof["lgb"][va_idx, h_idx] += p_va / len(SEEDS)
            raw_prob_test["lgb"][:, h_idx] += p_te / (len(SEEDS) * FOLDS)

source_oof = {}
source_test = {}

for name in ["cb", "xgb", "lgb", "dh"]:
    source_oof[name] = np.zeros((n_train, 4), dtype=float)
    source_test[name] = np.zeros((n_test, 4), dtype=float)
    for h_idx, h in enumerate(HORIZONS):
        valid, y = horizon_labels(time, event, h)
        cal = fit_platt(raw_prob_oof[name][:, h_idx], y, valid)
        source_oof[name][:, h_idx] = clip01(apply_calibrator(cal, raw_prob_oof[name][:, h_idx]))
        source_test[name][:, h_idx] = clip01(apply_calibrator(cal, raw_prob_test[name][:, h_idx]))

for name in ["cox_cb", "cox_xgb"]:
    source_oof[name] = np.zeros((n_train, 4), dtype=float)
    source_test[name] = np.zeros((n_test, 4), dtype=float)
    for h_idx, h in enumerate(HORIZONS):
        valid, y = horizon_labels(time, event, h)
        cal = fit_platt(raw_risk_oof[name], y, valid)
        source_oof[name][:, h_idx] = clip01(apply_calibrator(cal, raw_risk_oof[name]))
        source_test[name][:, h_idx] = clip01(apply_calibrator(cal, raw_risk_test[name]))

all_sources = ["cb", "xgb", "lgb", "dh", "cox_cb", "cox_xgb"]
stack_oof = np.zeros((n_train, 4), dtype=float)
stack_test = np.zeros((n_test, 4), dtype=float)
blend_weights = {}

for h_idx, h in enumerate(HORIZONS):
    valid, y = horizon_labels(time, event, h)
    Z_oof = np.column_stack([source_oof[s][:, h_idx] for s in all_sources])
    Z_test = np.column_stack([source_test[s][:, h_idx] for s in all_sources])
    w = fit_simplex_blend(Z_oof[valid], y[valid])
    blend_weights[f"h{h}"] = {src: float(weight) for src, weight in zip(all_sources, w)}
    stack_oof[:, h_idx] = clip01(Z_oof @ w)
    stack_test[:, h_idx] = clip01(Z_test @ w)

for h_idx, h in enumerate(HORIZONS):
    valid, y = horizon_labels(time, event, h)
    cal = fit_platt(stack_oof[:, h_idx], y, valid)
    stack_oof[:, h_idx] = clip01(apply_calibrator(cal, stack_oof[:, h_idx]))
    stack_test[:, h_idx] = clip01(apply_calibrator(cal, stack_test[:, h_idx]))

stack_oof[:, 1:] = np.maximum.accumulate(clip01(stack_oof[:, 1:]), axis=1)
stack_test[:, 1:] = np.maximum.accumulate(clip01(stack_test[:, 1:]), axis=1)

risk_components_oof = np.column_stack([
    rank_norm(raw_risk_oof["cox_cb"]),
    rank_norm(raw_risk_oof["cox_xgb"]),
    rank_norm(stack_oof[:, 0]),
    rank_norm(stack_oof[:, 1]),
    rank_norm(stack_oof[:, 2]),
    rank_norm(source_oof["dh"][:, 0]),
])

risk_components_test = np.column_stack([
    rank_norm(raw_risk_test["cox_cb"]),
    rank_norm(raw_risk_test["cox_xgb"]),
    rank_norm(stack_test[:, 0]),
    rank_norm(stack_test[:, 1]),
    rank_norm(stack_test[:, 2]),
    rank_norm(source_test["dh"][:, 0]),
])

rng = np.random.default_rng(42)
best_c = -1.0
best_w = None
for _ in range(6000):
    w = rng.dirichlet(np.ones(risk_components_oof.shape[1]))
    risk_candidate = risk_components_oof @ w
    c = harrell_c_index(time, event, risk_candidate)
    if c > best_c:
        best_c = c
        best_w = w.copy()

risk_oof = risk_components_oof @ best_w
risk_test = risk_components_test @ best_w
stack_oof[:, 0] = make_prob12_from_risk(risk_oof, stack_oof[:, 1])
stack_test[:, 0] = make_prob12_from_risk(risk_test, stack_test[:, 1])

internal_oof_score, internal_oof_cidx, internal_oof_wb = hybrid_score(
    time,
    event,
    stack_oof[:, 0],
    stack_oof[:, 1],
    stack_oof[:, 2],
    stack_oof[:, 3],
)

print({
    "internal_oof_hybrid": float(internal_oof_score),
    "internal_oof_cindex": float(internal_oof_cidx),
    "internal_oof_weighted_brier": float(internal_oof_wb),
})

final_prob24 = stack_test[:, 1].copy()
final_prob48 = stack_test[:, 2].copy()
final_prob72 = stack_test[:, 3].copy()
final_risk = rank_norm(risk_test).copy()
external_info = None
external_members = None

if USE_EXTERNAL_PRIORS_IF_FOUND:
    external_info = build_external_prior_blend(test["event_id"].to_numpy())
    if external_info is not None:
        external_members = external_info.get("members")
        print("Found external prior submissions:", len(external_info["members"]))
        final_prob24 = opinion_pool(final_prob24, external_info["prob24"], internal_weight=0.60)
        final_prob48 = opinion_pool(final_prob48, external_info["prob48"], internal_weight=0.60)
        final_prob72 = opinion_pool(final_prob72, external_info["prob72"], internal_weight=0.60)
        final_risk = 0.70 * rank_norm(final_risk) + 0.30 * rank_norm(external_info["risk_rank"])
    else:
        print("No eligible external samplecsv_sub*.csv priors found; using internal stack only.")

final_matrix = np.column_stack([final_prob24, final_prob48, final_prob72])
final_matrix = np.maximum.accumulate(clip01(final_matrix), axis=1)
final_prob24 = final_matrix[:, 0]
final_prob48 = final_matrix[:, 1]
final_prob72 = final_matrix[:, 2]
final_prob12 = make_prob12_from_risk(final_risk, final_prob24)

submission = pd.DataFrame({
    "event_id": test["event_id"].to_numpy(),
    "prob_12h": final_prob12,
    "prob_24h": final_prob24,
    "prob_48h": final_prob48,
    "prob_72h": final_prob72,
})

submission = sample[["event_id"]].merge(submission, on="event_id", how="left", validate="1:1")

assert submission["event_id"].is_unique
assert submission.shape[0] == sample.shape[0]
assert set(submission.columns) == {"event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"}
assert not submission.isna().any().any()
assert ((submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] >= 0.0) & (submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] <= 1.0)).all().all()
assert (submission["prob_12h"] <= submission["prob_24h"] + 1e-12).all()
assert (submission["prob_24h"] <= submission["prob_48h"] + 1e-12).all()
assert (submission["prob_48h"] <= submission["prob_72h"] + 1e-12).all()

os.makedirs("/kaggle/working", exist_ok=True)
submission_path = "/kaggle/working/submission.csv"
weights_path = "/kaggle/working/blend_weights.json"
summary_path = "/kaggle/working/internal_oof_metrics.json"

submission.to_csv(submission_path, index=False)
with open(weights_path, "w", encoding="utf-8") as f:
    json.dump({
        "horizon_blend_weights": blend_weights,
        "risk_blend_weights": best_w.tolist(),
        "external_priors": external_members,
    }, f, ensure_ascii=False, indent=2)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({
        "internal_oof_hybrid": float(internal_oof_score),
        "internal_oof_cindex": float(internal_oof_cidx),
        "internal_oof_weighted_brier": float(internal_oof_wb),
    }, f, ensure_ascii=False, indent=2)

print(submission.head())
print("Saved:", submission_path)
print("Saved:", weights_path)
print("Saved:", summary_path)


train: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv
test: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv
sample: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
{'internal_oof_hybrid': 0.9645502421201841, 'internal_oof_cindex': 0.9388042398145081, 'internal_oof_weighted_brier': 0.0244157568916689}
No eligible external samplecsv_sub*.csv priors found; using internal stack only.
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.050607  0.103761  0.118385  0.999999
1  13353600  0.086832  0.863005  0.866018  0.999999
2  13942327  0.048477  0.104008  0.118503  0.999999
3  16112781  0.092159  0.874290  0.874290  0.999999
4  17132808  0.067654  0.195695  0.201156  0.999999
Saved: /kaggle/working/submission.csv
Saved: /kaggle/working/blend_weights.json
Saved: /kaggle/working/internal_oof_metrics.json
